# Compare Pipes: Diameter vs. Head Loss

Reproduces the core comparison from the reference report: how does switching from a ½" to a 4" PVC distribution pipe affect head loss, velocity, and required pump power, for the same flow rate?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.simulation.scenario import PipeScenario
from src.simulation.sensitivity import sweep_parameter
from src.plots.plot_efficiency import head_loss_comparison_figure

## Baseline scenario and diameter sweep

In [ ]:
base = PipeScenario(
    diameter_m=0.0127,       # placeholder, overridden by the sweep
    flow_rate_m3s=0.0005,    # 0.5 L/s
    length_m=100.0,
    fittings={'elbow_90_standard': 4, 'gate_valve_open': 1},
)

diameters = np.linspace(0.0127, 0.1016, 12)  # 1/2" to 4"
df = sweep_parameter(base, 'diameter_m', diameters)
df[['diameter_m', 'velocity_m_s', 'reynolds', 'total_loss_m', 'shaft_power_W']]

## Direct ½" vs 4" comparison

Matches the two specific cases analyzed in the reference report.

In [ ]:
comparison = sweep_parameter(base, 'diameter_m', [0.0127, 0.1016])
comparison['label'] = ['1/2 inch', '4 inch']
comparison[['label', 'diameter_m', 'velocity_m_s', 'total_loss_m', 'pressure_drop_Pa', 'shaft_power_W']]

In [ ]:
ratio = comparison['total_loss_m'].iloc[0] / comparison['total_loss_m'].iloc[1]
print(f"Head loss is {ratio:,.0f}x higher with the 1/2-inch pipe "
      f"than the 4-inch pipe, for the same flow rate.")

## Visualize

In [ ]:
fig = head_loss_comparison_figure(df, 'diameter_m', x_label='Diameter (m)')
fig.show()

## Velocity (SNI 03-6481-2000 guideline check)

Recommended velocity range: 0.9-2.0 m/s. Pipes outside this range are flagged automatically by `run_simulation`.

In [ ]:
from src.simulation.scenario import run_simulation

for d, label in zip([0.0127, 0.1016], ['1/2 inch', '4 inch']):
    r = run_simulation(diameter_m=d, flow_rate_m3s=0.0005, length_m=100.0)
    print(f"{label}: v = {r.head_loss.velocity_m_s:.3f} m/s", r.velocity_warning or '(within SNI range)')